# Fundamentals 03 - Agent API

**Historia:** ya tienes tools. Ahora construyes agentes que deciden cuando usarlas.

Este notebook materializa dos rutas de `toolkit.agent(...)`:

1. **Agente determinista** con `engine="python-runtime"`: local, reproducible y sin proveedor externo.
2. **Agente LM** con `runtime(provider="auto")`: el mismo contrato se ejecuta con el backend disponible en el ambiente.

La intencion didactica es separar la API de Agentic Systems del backend real: el contrato, la policy, la validacion y `human_result(...)` son iguales en ambas rutas.

In [ ]:
import agentic_systems as toolkit

PRETTY = False

scheduler = toolkit.scheduler(
    timeout_s=60,
    max_retries=0,
    max_tool_calls=4,
    max_turns=8,
)

deterministic_runtime = toolkit.runtime(
    provider="python-runtime",
    model="local-python",
    region="local",
    scheduler=scheduler,
)

lm_runtime = toolkit.runtime(
    provider="auto",
    scheduler=scheduler,
)

toolkit.show({
    "deterministic_runtime": deterministic_runtime.describe(),
    "lm_runtime_auto_resolution": lm_runtime.describe(),
    "pretty_human_results": PRETTY,
}, title="Agent runtimes")

## Escenario didactico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para comparar la API sin cambiar de caso cada vez:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```

Resultado esperado: **42**.

## Parametros de `RunPolicy`

`RunPolicy` declara como debe comportarse una ejecucion antes de llamar al agente o al runtime. No es metadata decorativa: limita loops, define reparacion, controla trazas y hace que el resultado sea evaluable.

| Parametro | Que controla | Uso recomendado |
|---|---|---|
| `max_turns` | Numero maximo de turnos internos del agente. | Mantenerlo bajo en notebooks para evitar loops largos. |
| `max_tool_calls` | Numero maximo de llamadas a tools. | Declararlo cuando el ejercicio espera tools concretas. |
| `max_tokens` | Limite de tokens del modelo cuando el provider lo soporta. | util en providers LM; puede quedar `None` en `python-runtime`. |
| `temperature` | Aleatoriedad del modelo. | `0.0` para tutoriales reproducibles; `None` delega al provider. |
| `tool_choice` | Estrategia de seleccion de tools, por ejemplo `auto`. | `auto` cuando el agente decide; explicito cuando quieres forzar una tool. |
| `repair` | Permite reparacion automatica de salidas o tool calls invalidas. | `True` para UX robusta; `False` si quieres ver fallos crudos. |
| `max_repairs` | Maximo de intentos de reparacion. | `1` o `2` en tutoriales para mostrar control sin ocultar errores. |
| `finalize` | Que hacer al agotar turnos, por ejemplo `on_max_turns`. | Mantenerlo explicito en agentes LM evaluables. |
| `trace` | Nivel de trazabilidad (`compact`, `debug`, etc.). | `compact` para notebooks; `debug` solo para diagnostico. |
| `strict` | Si el contrato debe aplicarse de forma estricta. | `True` para ensenar API y evitar ambiguedad. |


In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

# La estructura solicitada por el usuario se materializa como datos simples.
# No es un parser ni una respuesta precocinada: solo representa la seccion `Dime:`.
REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

toolkit.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario didactico  visible")


## 1) Definir las tools del escenario didactico

Un agente, cuatro tools. El contrato va a exigir que el agente use todas para dejar evidencia paso a paso.

In [ ]:
@toolkit.tool
def sumar(a: int, b: int) -> dict:
    """Suma dos numeros."""
    return {"operation": "sumar", "result": a + b, "explanation": f"{a} + {b} = {a + b}"}


@toolkit.tool
def restar(a: int, b: int) -> dict:
    """Resta dos numeros."""
    return {"operation": "restar", "result": a - b, "explanation": f"{a} - {b} = {a - b}"}


@toolkit.tool
def multiplicar(a: int, b: int) -> dict:
    """Multiplica dos numeros."""
    return {"operation": "multiplicar", "result": a * b, "explanation": f"{a}  {b} = {a * b}"}


@toolkit.tool
def dividir(a: int, b: int) -> dict:
    """Divide dos numeros."""
    if b == 0:
        raise ValueError("No se puede dividir entre cero.")
    value = a / b
    result = int(value) if value.is_integer() else value
    return {"operation": "dividir", "result": result, "explanation": f"{a}  {b} = {result}"}


tools = [sumar, restar, multiplicar, dividir]
toolkit.show({"tools": [tool.name for tool in tools]})

## 2) Definir contrato y policy

El contrato declara que debe pasar; la policy limita como puede pasar.

In [ ]:
calculator_spec = toolkit.ContractPolicySpec(
    name="fundamentals.calculator_agent.default_problem",
    description="Resolver el escenario didactico con procedimiento y resultado final.",
    contract=toolkit.AgentContract(
        must_call=["sumar", "restar", "multiplicar", "dividir"],
        tool_expectation=toolkit.expect.all_of("sumar", "restar", "multiplicar", "dividir"),
        completion="when_required_tools_satisfied",
        failure_policy="no_unresolved",
        expected_tool_outputs={
            "sumar": {"result": 30},
            "restar": {"result": 21},
            "multiplicar": {"result": 84},
            "dividir": {"result": 42},
        },
    ),
    policy=toolkit.RunPolicy(max_turns=8, max_tool_calls=4, temperature=0.0, finalize="after_required_tools"),
)

toolkit.show({
    "spec": calculator_spec.describe(),
    "static_check": calculator_spec.check(available_tools=[tool.name for tool in tools]).to_dict(),
})

<!-- run-policy-parameters -->
## Como leer `RunPolicy`

`RunPolicy` es la politica de ejecucion de un agente. No define que hace el negocio; define cuanto puede intentar el agente, como usa tools y que tan estricta debe ser la validacion.

| Parametro | Que controla | Regla practica |
| --- | --- | --- |
| `max_turns` | Numero maximo de turnos del loop agente/modelo/tool. | Subelo si el agente necesita varios pasos; bajalo para cortar loops largos. |
| `max_tool_calls` | Limite total de ejecuciones de tools. `None` significa que no agrega un limite propio. | Usalo cuando el contrato exige una cantidad acotada de acciones. |
| `max_tokens` | Presupuesto de generacion que se pasa al provider cuando el backend lo soporta. | Dejalo en `None` para usar el default del runtime/provider. |
| `temperature` | Aleatoriedad de salida en providers LM. | `0.0` para evals reproducibles; `None` para usar el default del provider. |
| `tool_choice` | Preferencia de uso de tools en runtimes LM. | `auto` deja que el modelo decida; valores forzados dependen del provider. |
| `repair` | Permite reparar/reintentar llamadas invalidas de tools. | `True` para robustez; `False` para detectar fallos temprano. |
| `max_repairs` | Numero maximo de reparaciones permitidas. | Mantener bajo en produccion para evitar ciclos opacos. |
| `finalize` | Cuando sintetizar respuesta final. | `on_max_turns` cierra al agotar turnos; `after_required_tools` cierra al cumplir contrato; `never` no sintetiza. |
| `trace` | Nivel de traza guardada. | `compact` para notebooks; `full` para auditoria profunda. |
| `strict` | Severidad de validacion de contrato/policy. | `True` para produccion/evals; `False` solo para debug exploratorio. |

Los `mode` (`default`, `fast`, `eval`, `audit`, `debug`, `prod`) son presets de `RunPolicy`. El notebook los muestra como JSON para inspeccion, pero esta tabla explica como interpretar cada campo.


## 3) Crear un agente determinista con `python-runtime`

Este agente no usa LLM. El runtime local ejecuta un plan determinista contra las tools registradas. Sirve para pruebas unitarias, contratos y ejemplos que deben correr igual en cualquier maquina.

In [ ]:
instructions = """
Eres un agente calculadora.
Usa las tools disponibles para resolver el problema aritm?tico.
No hagas c?lculo mental cuando exista una tool adecuada.
Responde en espa?ol con:
- procedimiento
- resultado final
""".strip()

agent_controls = calculator_spec.agent_kwargs()

deterministic_agent = toolkit.agent(
    name="calculator_deterministic_agent",
    instructions=instructions,
    tools=tools,
    engine="python-runtime",
    runtime=deterministic_runtime,
    **agent_controls,
)

toolkit.show({
    "agent": deterministic_agent.info(),
    "contract_policy": calculator_spec.describe(),
    "agent_controls": agent_controls,
}, title="Agente determinista")

## 4) Ejecutar el agente determinista

La entrada es estructurada para que `python-runtime` pueda demostrar el contrato sin depender de interpretacion de lenguaje natural.

In [ ]:
deterministic_input = {"tool": "sumar", "input": {"a": 10, "b": 20}}

single_call_contract = toolkit.AgentContract(
    must_call=["sumar"],
    tool_expectation=toolkit.expect.exactly("sumar"),
    completion="when_required_tools_satisfied",
)

deterministic_agent.contract = single_call_contract

deterministic_result = deterministic_agent.run(deterministic_input, mode="eval")
deterministic_result.validation = deterministic_result.validate(single_call_contract).to_dict()

toolkit.human_result(
    deterministic_result,
    title="Human result - agente determinista - python-runtime",
    expected_tools=single_call_contract.tool_expectation,
    pretty=PRETTY,
)

## 5) Crear un agente LM con `runtime(provider="auto")`

`auto` no significa magia oculta: `RuntimeConfig.describe()` declara que provider selecciono por senales del ambiente.

- En VS Code local o sandbox, `auto` selecciona el backend disponible por senales del ambiente.
- Si no hay senales, el notebook salta esta ejecucion y deja explicita la razon.
- El codigo del agente no cambia cuando cambias de backend.


In [ ]:
lm_resolution = lm_runtime.describe()
lm_provider = lm_resolution["selected_provider"]

lm_system = toolkit.AgenticSystem(
    model=lm_runtime.model_id or toolkit.default_model_id(),
    region=lm_runtime.region_name or toolkit.default_region(),
    runtime=lm_runtime,
)

lm_agent = lm_system.agent(
    name="calculator_lm_runtime_agent",
    instructions=instructions,
    tools=tools,
    runtime=lm_runtime,
    contract=calculator_spec.contract,
    policy=calculator_spec.policy,
)

runtime_diagnostics = {
    "runtime_resolution": lm_resolution,
    "lm_agent": lm_agent.info(),
}
if lm_provider == "bedrock-runtime":
    runtime_diagnostics["boto3_session"] = toolkit.boto3_session_snapshot(lm_runtime.region_name or toolkit.default_region())

toolkit.show(runtime_diagnostics, title="Agente LM con runtime seleccionado")

## 6) Ejecutar el agente LM si hay provider disponible

Esta celda mantiene el tutorial agnostico: no fuerza OpenAI ni Bedrock. Si `auto` no encuentra configuracion, se reporta como skip controlado.


In [ ]:
if lm_provider == "auto":
    toolkit.show({
        "status": "skipped",
        "reason": lm_resolution["reason"],
        "how_to_enable": "Configura OPENAI_API_KEY o credenciales/configuracion Bedrock antes de abrir el kernel.",
    }, title="Agente LM saltado")
else:
    lm_result = lm_agent.run(USER_PROMPT, mode="eval")
    lm_result.validation = lm_result.validate(calculator_spec.contract).to_dict()

    toolkit.human_result(
        lm_result,
        title=f"Human result - agente LM - {lm_provider}",
        expected_tools=calculator_spec.contract.tool_expectation,
        pretty=PRETTY,
    )


## Lo importante

- `toolkit.agent(...)` no oculta el contrato.
- `engine="python-runtime"` es la ruta determinista local.
- `runtime(provider="auto")` es la ruta LM agnostica al backend.
- `AgentContract` define las tools esperadas.
- `RunPolicy` controla presupuesto y finalizacion.
- `RunResult.validate(...)` separa ejecucion de validacion.
- `human_result(...)` renderiza, no decide la verdad del dominio.

## Coverage API de este notebook

Esta tabla deja explicito que parte de Agentic Systems queda materializada aqui.

In [ ]:
api_coverage = [
    {
        "api": "toolkit.agent",
        "description": "Crea agentes canonicos con contrato, tools y runtime declarados."
    },
    {
        "api": 'engine="python-runtime"',
        "description": "Ejecuta agentes deterministas locales sin proveedor externo."
    },
    {
        "api": 'runtime(provider="auto")',
        "description": "Ejecuta agentes LM con seleccion automatica de provider segun el ambiente."
    },
    {
        "api": "AgenticSystem(...).agent",
        "description": "Vincula un agente LM a un system que puede resolver providers reales."
    },
    {
        "api": "AgentContract",
        "description": "Define que tools y salidas son obligatorias para el agente."
    },
    {
        "api": "RunPolicy",
        "description": "Controla el comportamiento de ejecucion del agente."
    },
    {
        "api": "RunResult.validate",
        "description": "Valida el resultado contra el contrato despues de ejecutar."
    },
    {
        "api": "human_result",
        "description": "Renderiza la salida humana sin perder la evidencia interna."
    },
    {
        "api": "default arithmetic prompt",
        "description": "Usa el mismo prompt base para comparar comportamiento entre notebooks."
    }
]

toolkit.show({'notebook': '03_agent_api.ipynb', 'api_coverage': api_coverage})

## Simbolos API explicados

Este notebook se alinea con `docs/API.md` y ensena estos simbolos publicos:

- `Agent / toolkit.agent`: Primitiva que convierte contexto en acciones.
- `python-runtime`: Engine determinista para tools locales.
- `provider="auto"`: Seleccion agnostica del backend LM disponible.
- `AgentContract / ContractPolicySpec / RunPolicy`: Contrato y politica del agente.
- `validate_contract_policy`: Validacion publica del contrato/policy.
- `human_result`: Salida humana de ejecuciones reales.

